<a href="https://colab.research.google.com/github/eeeewyz/Audiomodel/blob/main/Matcha_TTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matcha-TTS: A fast TTS architecture with conditional flow matching
---
[Shivam Mehta](https://www.kth.se/profile/smehta), [Ruibo Tu](https://www.kth.se/profile/ruibo), [Jonas Beskow](https://www.kth.se/profile/beskow), [Éva Székely](https://www.kth.se/profile/szekely), and [Gustav Eje Henter](https://people.kth.se/~ghe/)

We introduce Matcha-TTS, a new encoder-decoder architecture for speedy TTS acoustic modelling, trained using optimal-transport conditional flow matching (OT-CFM). This yields an ODE-based decoder capable of high output quality in fewer synthesis steps than models trained using score matching. Careful design choices additionally ensure each synthesis step is fast to run. The method is probabilistic, non-autoregressive, and learns to speak from scratch without external alignments. Compared to strong pre-trained baseline models, the Matcha-TTS system has the smallest memory footprint, rivals the speed of the fastest models on long utterances, and attains the highest mean opinion score in a listening test.

Demo Page: https://shivammehta25.github.io/Matcha-TTS \
Code: https://github.com/shivammehta25/Matcha-TTS

## 0. Colab 环境安装（v3：解决 NumPy/HiFi-GAN 导入错误）

**第一次运行：从 0-A 开始依次执行。** 0-F 会在安装成功后**自动重启 Python 运行时一次**（这是修复混合 NumPy 文件的必要步骤）；Colab 重新连接后，请再次从 0-A 开始运行，安装步骤会根据标记自动跳过，然后继续第 1 部分。

**如果之前运行过 v1/v2：请先点击「运行时 → 断开连接并删除运行时」**，不要仅仅再次运行导入 Cell。第一次安装可能需要几分钟。

- 目标：Python 3.10–3.13，Colab 自带 Torch/CUDA，匹配的 NumPy + Cython + Matcha-TTS。
- 本次错误的检查点：`numpy.ufunc` 必须允许访问与更新 `__module__`；发现混合版本就立刻报告。
- **重要限制：**本文件经过静态检查和本地模拟，但没有真实 Colab GPU 环境供完整安装/推理测试；不要把本地检查误认为云端已验证成功。

In [1]:
# 0-A｜检查 Colab 环境 + 是否完成安装（此 Cell 不 import NumPy/Torch）
import sys
from pathlib import Path
from importlib import metadata

SETUP_STATE = Path('/content/.matcha_colab_v3_state')
SETUP_VERSION = 'v3_numpy_consistent_20261015'
INSTALLED_AND_RESTARTED = (SETUP_STATE.exists() and SETUP_STATE.read_text().strip() == SETUP_VERSION)
NUMPY_VERSION = '2.2.6' if sys.version_info >= (3, 13) else '1.26.4'

print('Python:', sys.version.split()[0])
print('安装目标 NumPy:', NUMPY_VERSION)
print('当前环境是否已安装并重启：', INSTALLED_AND_RESTARTED)
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('本文件目前面向 Colab Python 3.10～3.13；其他版本尚未验证。')
try:
    print('当前 PyTorch:', metadata.version('torch'))
except metadata.PackageNotFoundError as exc:
    raise RuntimeError('请使用自带 PyTorch 的 Colab Python 运行时') from exc

# 用户如果之前用过 v2，建议先「断开连接并删除运行时」彻底清理。
if not INSTALLED_AND_RESTARTED and ('numpy' in sys.modules or 'torch' in sys.modules):
    print('提示：Colab 可能已预加载 NumPy/Torch；本文件将在安装结束后自动重启内核。')

Python: 3.13.15
安装目标 NumPy: 2.2.6
当前环境是否已安装并重启： True
当前 PyTorch: 2.11.0+cu130


In [2]:
# 0-B｜安装 Linux 系统工具：英语音素转换 + WAV + 编译工具
# 安装后保存在当前 Colab 运行时；已安装并重启则跳过
import subprocess
if not INSTALLED_AND_RESTARTED:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq',
                    'espeak-ng', 'libsndfile1', 'build-essential'], check=True)
else:
    print('✓ 之前已安装系统依赖，跳过')

✓ 之前已安装系统依赖，跳过


In [3]:
# 0-C｜固定 NumPy / Cython / 构建工具的版本
# --force-reinstall：清除上一版本文件，防止 Python 文件与二进制模块混合
# --only-binary=:all:：NumPy 必须使用 Python 版本匹配的预编译 wheel
if not INSTALLED_AND_RESTARTED:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                    '--force-reinstall', '--only-binary=:all:', f'numpy=={NUMPY_VERSION}'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--prefer-binary',
                    'Cython==3.0.12', 'setuptools==80.9.0',
                    'wheel==0.45.1', 'packaging>=24,<26'], check=True)
else:
    print('✓ NumPy / Cython 已安装，跳过')

✓ NumPy / Cython 已安装，跳过


In [4]:
# 0-D｜统一安装 Matcha 运行依赖（保留 Colab 自带 Torch/CUDA）
# 把 NumPy 的精确版本加入同一轮求解，防止依赖安装时再次升降级
# 只安装 Matcha 所需的功能库，不额外安装另一个 torch
RUNTIME_PACKAGES = [
    f'numpy=={NUMPY_VERSION}',        # 固定数值计算/NumPy C 扩展版本
    'lightning==2.6.0',              # Lightning checkpoint 加载
    'torchmetrics==1.8.2',          # Lightning 的指标依赖
    'hydra-core==1.3.2',            # 原项目配置系统
    'omegaconf==2.3.0',             # 配置对象
    'rich==13.9.4',                 # 日志显示
    'rootutils>=1.0,<2',            # 项目路径工具
    'conformer==0.3.2',             # Matcha Decoder 组件
    'einops==0.8.1',                # Tensor 维度转换
    'diffusers==0.28.2',           # Matcha 内部使用的模型层
    'huggingface-hub==0.25.2',      # 与上方 Diffusers 版本匹配
    'transformers==4.48.3',        # Hugging Face 接口
    'tokenizers==0.21.4',          # 使用预编译 wheel，无需 Rust
    'phonemizer==3.3.0',           # 英语文本→音素
    'inflect==7.5.0',              # 文本标准化
    'Unidecode==1.4.0',            # 字符标准化
    'gdown==5.2.0',                # 下载工具（兼容原项目）
    'wget==3.2',                   # 下载工具（兼容原项目）
    'soundfile==0.12.1',           # WAV 读写
    'librosa==0.10.2.post1',       # 音频处理
    'matplotlib>=3.8,<4',          # 频谱可视化
    'tqdm==4.67.1',               # 进度条
]
if not INSTALLED_AND_RESTARTED:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--prefer-binary',
                    *RUNTIME_PACKAGES], check=True)
    print('✓ 运行依赖安装成功')
else:
    print('✓ 运行依赖已安装，跳过')

✓ 运行依赖已安装，跳过


In [5]:
# 0-E｜安装 Matcha-TTS 源代码并在【全新 Python 子进程】检查版本
# --no-build-isolation：规避官方 pyproject.toml 的旧版 NumPy/Cython 锁定
# --no-deps：避免原仓库的 requirements 自动更换统一依赖
if not INSTALLED_AND_RESTARTED:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                    '--no-build-isolation', '--no-deps', 'matcha-tts==0.0.7.2'], check=True)

# 下列检查发生在新的 Python 进程，不使用当前 Notebook 缓存的 NumPy
CHECK_NUMPY = r"""
import numpy as np
import importlib
if np.__version__.startswith("2."):
    importlib.import_module("numpy._core.strings")
else:
    importlib.import_module("numpy.core.defchararray")
from importlib.metadata import version
expected = __import__("sys").argv[1]
assert np.__version__ == expected, (np.__version__, expected)
assert version('numpy') == expected, (version('numpy'), expected)
assert hasattr(np.add, '__module__'), 'NumPy ufunc 缺少 __module__'
# NumPy 2.x 的 ufunc 支持修改 __module__，必要时验证底层 C 扩展
if np.__version__.startswith('2.'):
    old = np.add.__module__
    np.add.__module__ = old
print('NumPy 健康检查通过:', np.__version__, np.__file__)
"""
subprocess.run([sys.executable, '-c', CHECK_NUMPY, NUMPY_VERSION], check=True)
print('✓ NumPy Python / C 扩展检查通过')

✓ NumPy Python / C 扩展检查通过


In [6]:
# 0-F｜首次安装成功后，自动重启 Python 内核一次
# 安装过程中如果 NumPy 曾被导入，旧的 .so 二进制模块可能残留在内存中。
# 必须重启，而不能只重新运行 1-B Import Cell。
import os
import signal
if not INSTALLED_AND_RESTARTED:
    SETUP_STATE.parent.mkdir(parents=True, exist_ok=True)
    SETUP_STATE.write_text(SETUP_VERSION)
    print('✓ 安装完成。即将自动重启 Python 运行时。', flush=True)
    print('Colab 重新连接后请再次从 0-A 开始运行；安装将自动跳过。', flush=True)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('✓ 安装完成并已重启；无需再次重启')

✓ 安装完成并已重启；无需再次重启


In [7]:
# 0-G｜在当前 Colab 内核中做【真正的】导入检查
# 先确认 NumPy 的 Python 源码与编译扩展一致，再依次检查项目模块
# 本 Cell 必须通过后才能继续 1-A、1-B。
import numpy as np
from importlib import metadata
# NumPy 2.x：主动导入截图中发生异常的 strings 模块
import importlib
if np.__version__.startswith("2."):
    _np_strings = importlib.import_module("numpy._core.strings")
else:
    _np_strings = importlib.import_module("numpy.core.defchararray")
assert np.__version__ == NUMPY_VERSION, (
    f'NumPy 内存中版本 {np.__version__} 与安装目标 {NUMPY_VERSION} 不一致！请删除 Colab 运行时后从头执行。'
)
assert metadata.version('numpy') == NUMPY_VERSION, 'NumPy 磁盘安装版本与目标不同'
assert hasattr(np.add, '__module__'), 'NumPy ufunc 模块异常，请删除运行时后重试'

# 这四项正是 1-B 导入 HiFi-GAN 时用到的组件
from matcha.hifigan.config import v1 as _test_config
from matcha.hifigan.denoiser import Denoiser as _test_denoiser
from matcha.hifigan.env import AttrDict as _test_attrdict
from matcha.hifigan.models import Generator as _test_generator
from matcha.models.matcha_tts import MatchaTTS as _test_matcha
from matcha.text import text_to_sequence as _test_text

print('✓ NumPy、HiFi-GAN、Matcha-TTS 真实导入成功')
print('NumPy:', np.__version__, '| NumPy 路径:', np.__file__)
print('Matcha 包:', metadata.version('matcha-tts'))
print('英语音素 token 数:', len(_test_text('Hello world.', ['english_cleaners2'])[0]))

✓ NumPy、HiFi-GAN、Matcha-TTS 真实导入成功
NumPy: 2.2.6 | NumPy 路径: /usr/local/lib/python3.13/dist-packages/numpy/__init__.py
Matcha 包: 0.0.7.2
英语音素 token 数: 14


## Import libraries（导入库）

和原 Notebook 相同的 import，拆成 Python 工具、HiFi-GAN、Matcha-TTS 三组。

In [8]:
# 1-A｜Python 与音频数据工具
# 必须先通过 0-G；避免在 NumPy 尚未稳定时直接导入 HiFi-GAN
if 'SETUP_VERSION' not in globals() or not INSTALLED_AND_RESTARTED:
    raise RuntimeError('请先从 0-A 执行，首次安装后等待自动重启，再运行 0-G。')
import datetime as dt         # 记录合成开始时间，用于计算 RTF
from pathlib import Path     # 管理模型、输出文件的路径

import IPython.display as ipd # 在 Colab 中嵌入音频播放器
import numpy as np           # 数值计算、保存 Mel 特征数组
import soundfile as sf       # 将合成音频保存为 .wav
import torch                 # PyTorch 张量与模型推理
from tqdm.auto import tqdm   # 批量合成时显示进度条

# Colab 默认内嵌显示 Matplotlib 图像；这里不执行 %matplotlib inline


In [9]:
# 1-B｜HiFi-GAN（Vocoder：Mel-spectrogram → Waveform）
# 0-G 若未通过，不要跳过它直接运行本 Cell
from matcha.hifigan.config import v1           # HiFi-GAN v1 的网络配置
from matcha.hifigan.denoiser import Denoiser   # 减弱 Vocoder 合成噪声
from matcha.hifigan.env import AttrDict        # 将配置字典转换为属性访问形式
from matcha.hifigan.models import Generator as HiFiGAN  # HiFi-GAN 生成器
print('✓ HiFi-GAN import 成功')


✓ HiFi-GAN import 成功


In [10]:
# 1-C｜Matcha-TTS（Acoustic Model：Text → Mel-spectrogram）
from matcha.models.matcha_tts import MatchaTTS               # 主模型 / Conditional Flow Matching
from matcha.text import sequence_to_text, text_to_sequence   # 文字、音素与 token ID 转换
from matcha.utils.model import denormalize                   # 将归一化 Mel 还原（原 Notebook 保留）
from matcha.utils.utils import get_user_data_dir, intersperse
# get_user_data_dir：获得 Matcha 模型数据目录
# intersperse：在音素序列之间插入 blank ID（0）

print("✓ 所有原始 import 已完成")

✓ 所有原始 import 已完成


In [11]:
# 1-D｜选择推理设备（保留原来的 device 变量）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("当前运行设备：", device)
print("PyTorch 版本：", torch.__version__)
if device.type == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
else:
    print("提示：CPU 也能生成语音，但速度明显慢于 GPU；推荐在 Colab 选择 T4 GPU。")

当前运行设备： cuda
PyTorch 版本： 2.11.0+cu130
GPU： Tesla T4


## Filepaths（文件路径）

In [12]:
# 2｜模型权重路径及生成文件目录（保留原变量名）
MATCHA_CHECKPOINT = get_user_data_dir() / "matcha_ljspeech.ckpt"
# Matcha-TTS 预训练权重：Text → Mel-spectrogram

HIFIGAN_CHECKPOINT = get_user_data_dir() / "hifigan_T2_v1"
# HiFi-GAN Vocoder 预训练权重：Mel-spectrogram → Waveform

OUTPUT_FOLDER = "synth_output"
# 最终生成音频、Mel .npy 的保存目录
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

print("Matcha 权重位置：", MATCHA_CHECKPOINT)
print("HiFi-GAN 权重位置：", HIFIGAN_CHECKPOINT)
print("输出文件夹：", Path(OUTPUT_FOLDER).resolve())

Matcha 权重位置： /root/.local/share/matcha_tts/matcha_ljspeech.ckpt
HiFi-GAN 权重位置： /root/.local/share/matcha_tts/hifigan_T2_v1
输出文件夹： /content/synth_output


### Download pretrained checkpoints（新增，必须运行一次）

**原 Notebook 直接读取模型文件，没有下载步骤**。下面使用官方仓库 `matcha/cli.py` 中的 GitHub Releases 地址，下载到上面的两个固定路径；再次运行会自动跳过已存在的有效文件。

In [13]:
# 2-B｜下载官方 Matcha-TTS 与 HiFi-GAN 权重
# 下载只需要在每个新的 Colab 运行时进行一次（Colab 本地文件是临时的）
import shutil
import urllib.request

MATCHA_URL = (
    "https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/"
    "download/v1.0/matcha_ljspeech.ckpt"
)
HIFIGAN_URL = (
    "https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/"
    "download/v1.0/generator_v1"
)

def download_if_missing(url, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    # 过小文件通常是下载错误页面，不能当模型权重
    if target.exists() and target.stat().st_size > 1_000_000:
        print(f"✓ 已存在：{target.name}（{target.stat().st_size / 1e6:.1f} MB）")
        return

    temp_path = target.with_name(target.name + ".download")
    print("正在下载：", target.name)
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=120) as response, open(temp_path, "wb") as output_file:
            shutil.copyfileobj(response, output_file)
        if temp_path.stat().st_size <= 1_000_000:
            raise RuntimeError(f"下载文件异常小：{temp_path.stat().st_size} bytes")
        temp_path.replace(target)  # 完整下载后，才将临时文件替换为权重文件
        print(f"✓ 下载完成：{target}（{target.stat().st_size / 1e6:.1f} MB）")
    except Exception:
        temp_path.unlink(missing_ok=True)  # 失败时清理不完整的下载
        raise

download_if_missing(MATCHA_URL, MATCHA_CHECKPOINT)
download_if_missing(HIFIGAN_URL, HIFIGAN_CHECKPOINT)
assert MATCHA_CHECKPOINT.is_file() and HIFIGAN_CHECKPOINT.is_file()
print("✓ 两个预训练权重均已准备好")

正在下载： matcha_ljspeech.ckpt
✓ 下载完成：/root/.local/share/matcha_tts/matcha_ljspeech.ckpt（218.9 MB）
正在下载： hifigan_T2_v1
✓ 下载完成：/root/.local/share/matcha_tts/hifigan_T2_v1（55.8 MB）
✓ 两个预训练权重均已准备好


## Load Matcha-TTS（加载 Acoustic Model）

In [14]:
# 3｜加载 Matcha-TTS：预训练 Text → Mel 模型
def load_model(checkpoint_path):
    # 原官方发布的 checkpoint 为可信来源；可能包含 Lightning/OmegaConf 对象
    # PyTorch 2.6+ 将 weights_only 默认为 True，因此这里显式允许加载完整 checkpoint
    # 注意：weights_only=False 仅适用于可信来源的权重文件！
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Matcha 权重不存在，请先运行 2-B：{checkpoint_path}")
    model = MatchaTTS.load_from_checkpoint(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    model = model.to(device)  # 原 Notebook 未显式迁移到 GPU，Colab 使用 T4 时需要补上
    model.eval()             # 推理模式，禁用 Dropout 等训练行为
    return model

# 原来的参数总量统计逻辑
count_params = lambda x: f"{sum(p.numel() for p in x.parameters()):,}"

model = load_model(MATCHA_CHECKPOINT)
print(f"Model loaded! Parameter count: {count_params(model)}")
print("Matcha 模型所在设备：", next(model.parameters()).device)

/usr/local/lib/python3.13/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


Model loaded! Parameter count: 18,204,193
Matcha 模型所在设备： cuda:0


## Load HiFi-GAN (Vocoder)（加载声码器）

In [15]:
# 4｜加载 HiFi-GAN：Mel-spectrogram → Waveform
def load_vocoder(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"HiFi-GAN 权重不存在，请先运行 2-B：{checkpoint_path}")
    h = AttrDict(v1)                       # 读取官方 HiFi-GAN v1 模型超参数
    hifigan = HiFiGAN(h).to(device)         # 实例化并放到 GPU/CPU
    # 此文件来自上面固定的官方发布链接；只对可信权重允许 weights_only=False
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    hifigan.load_state_dict(checkpoint["generator"])
    hifigan.eval()                         # 推理模式
    hifigan.remove_weight_norm()           # 移除训练用 Weight Normalization，加速推理
    return hifigan

vocoder = load_vocoder(HIFIGAN_CHECKPOINT)
denoiser = Denoiser(vocoder, mode="zeros")  # 用于减弱 Vocoder 的背景噪声
print("✓ HiFi-GAN 与 Denoiser 已加载")

/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...
✓ HiFi-GAN 与 Denoiser 已加载


### Helper functions to synthesise（语音合成辅助函数）

In [16]:
# 5-A｜文本预处理：Text → Phonemes → Token IDs
@torch.inference_mode()
def process_text(text: str):
    # english_cleaners2：英语文本归一化与音素转换
    # [0] 取 token ID 序列；intersperse(..., 0) 在音素 ID 之间加入 blank
    tokens = text_to_sequence(text, ["english_cleaners2"])[0]
    x = torch.tensor(
        intersperse(tokens, 0), dtype=torch.long, device=device
    )[None]  # shape: (batch=1, text_length)
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=device)
    x_phones = sequence_to_text(x.squeeze(0).tolist())  # 供学习观察的音素字符串
    return {
        "x_orig": text,     # 原始输入文字
        "x": x,             # 模型用 Text Token IDs
        "x_lengths": x_lengths,
        "x_phones": x_phones,
    }

In [17]:
# 5-B｜Matcha-TTS 合成 Mel（对应原来的 synthesise 函数）
@torch.inference_mode()
def synthesise(text, spks=None):
    text_processed = process_text(text)
    start_t = dt.datetime.now()          # 计时：Matcha + HiFi-GAN 总耗时
    output = model.synthesise(
        text_processed["x"],
        text_processed["x_lengths"],
        n_timesteps=n_timesteps,          # Euler ODE Solver 的步数
        temperature=temperature,          # 初始随机噪声的标准差缩放
        spks=spks,                        # LJSpeech 为单说话人，默认 None
        length_scale=length_scale,        # >1 更慢、<1 更快
    )
    # 将原文本、音素、开始时间合并到模型输出字典
    output.update({"start_t": start_t, **text_processed})
    return output

In [18]:
# 5-C｜HiFi-GAN 合成 Waveform，并用 Denoiser 降噪
@torch.inference_mode()
def to_waveform(mel, vocoder):
    audio = vocoder(mel).clamp(-1, 1)                   # Mel → 音频波形，幅值裁剪到 [-1,1]
    audio = denoiser(audio.squeeze(0), strength=0.00025).cpu().squeeze()
    return audio.cpu().squeeze()                       # 返回 CPU Tensor，方便播放与保存

In [19]:
# 5-D｜保存 Mel 特征（.npy）和最终音频（.wav）
def save_to_folder(filename: str, output: dict, folder: str):
    folder = Path(folder)
    folder.mkdir(exist_ok=True, parents=True)
    np.save(folder / f"{filename}", output["mel"].detach().cpu().numpy())
    # 22050 Hz 是原模型和 HiFi-GAN 的采样率；PCM_24 表示 24-bit WAV
    waveform = output["waveform"].detach().cpu().numpy()
    sf.write(folder / f"{filename}.wav", waveform, 22050, "PCM_24")

## Setup text to synthesise（要合成的文本）

In [20]:
# 6｜修改下面文本即可合成其他英语句子（原示例保持不变）
texts = [
    "The Secret Service believed that it was very doubtful that any President would ride regularly in a vehicle with a fixed top, even though transparent."
]
# 本笔记使用的是 LJSpeech 英语单说话人模型，不是多语言或中文 TTS。

### Hyperparameters（推理超参数）

In [21]:
# 7｜Matcha-TTS 关键推理参数（保留原默认值）
# Euler ODE Solver steps：速度场积分步数，越大通常越耗时
n_timesteps = 10

# Speaking rate scale：控制语音时长；>1 偏慢，<1 偏快
length_scale = 1.0

# Sampling temperature：采样噪声的缩放幅度，不是训练的温度
# 数值越大，音频随机性通常越强（并非越高质量）
temperature = 0.667

### 7-B. 最小端到端测试（建议先运行）

先用一句很短的英语检验 **Text → Phonemes → Matcha Mel → HiFi-GAN Waveform**，验证真实模型可以推理、Tensor 形状和音频数值正常，然后再运行原来的长文本。该测试不修改原 `texts`。

In [22]:
# 7-B｜推理冒烟测试：不用长文本，先运行最小完整流程
# 成功输出：Mel (1, 80, T)、Waveform (N,) 且含有限数值
with torch.inference_mode():
    _smoke = synthesise("Hello world.")
    _smoke["waveform"] = to_waveform(_smoke["mel"], vocoder)

assert _smoke["mel"].ndim == 3, f"Mel 维度错误：{_smoke['mel'].shape}"
assert _smoke["mel"].shape[1] == 80, f"预期 80-bin Mel，得到：{_smoke['mel'].shape}"
assert _smoke["waveform"].ndim == 1 and _smoke["waveform"].numel() > 0
assert torch.isfinite(_smoke["mel"]).all(), "Mel 存在 NaN / Inf"
assert torch.isfinite(_smoke["waveform"]).all(), "Waveform 存在 NaN / Inf"
print("✓ 端到端推理通过")
print("Mel shape:", tuple(_smoke["mel"].shape))
print("Waveform shape:", tuple(_smoke["waveform"].shape))
print("Waveform duration (s):", round(_smoke["waveform"].numel() / 22050, 2))
ipd.display(ipd.Audio(_smoke["waveform"].numpy(), rate=22050))


✓ 端到端推理通过
Mel shape: (1, 80, 81)
Waveform shape: (20736,)
Waveform duration (s): 0.94


## Synthesis（合成语音）

In [23]:
# 8-A｜批量合成：Matcha 预测 Mel → HiFi-GAN 生成 Waveform
outputs, rtfs = [], []  # 与原 Notebook 保持一致：后续分析可以使用 outputs、rtfs
rtfs_w = []              # 包括 HiFi-GAN 的 Real-Time Factor

for i, text in enumerate(tqdm(texts)):
    output = synthesise(text)                             # 第一阶段：生成 Mel 特征
    output["waveform"] = to_waveform(output["mel"], vocoder)  # 第二阶段：生成 Waveform

    # RTF = 总计算耗时 / 实际音频时长；越小，合成越快
    elapsed = (dt.datetime.now() - output["start_t"]).total_seconds()
    rtf_w = elapsed * 22050 / output["waveform"].shape[-1]

    # 与原 Notebook 保持一致的文本、音素、RTF 打印信息
    print(f"{'*' * 53}")
    print(f"Input text - {i}")
    print(f"{'-' * 53}")
    print(output["x_orig"])
    print(f"{'*' * 53}")
    print(f"Phonetised text - {i}")
    print(f"{'-' * 53}")
    print(output["x_phones"])
    print(f"{'*' * 53}")
    print(f"RTF:\t\t{output['rtf']:.6f}")
    print(f"RTF Waveform:\t{rtf_w:.6f}")
    rtfs.append(output["rtf"])
    rtfs_w.append(rtf_w)

    # Colab 直接播放合成的 .wav（waveform 为 CPU Tensor）
    ipd.display(ipd.Audio(output["waveform"].numpy(), rate=22050))
    save_to_folder(i, output, OUTPUT_FOLDER)  # 保存 i.wav、i.npy
    outputs.append(output)                     # 保存本轮输出供后续研究

print("✓ 语音合成完成；文件位于：", Path(OUTPUT_FOLDER).resolve())

  0%|          | 0/1 [00:00<?, ?it/s]

*****************************************************
Input text - 0
-----------------------------------------------------
The Secret Service believed that it was very doubtful that any President would ride regularly in a vehicle with a fixed top, even though transparent.
*****************************************************
Phonetised text - 0
-----------------------------------------------------
_ð_ə_ _s_ˈ_i_ː_k_ɹ_ᵻ_t_ _s_ˈ_ɜ_ː_v_ɪ_s_ _b_ᵻ_l_ˈ_i_ː_v_d_ _ð_ˌ_ɐ_ɾ_ɪ_t_ _w_ʌ_z_ _v_ˈ_ɛ_ɹ_i_ _d_ˈ_a_ʊ_t_f_ə_l_ _ð_æ_t_ _ˌ_ɛ_n_i_ _p_ɹ_ˈ_ɛ_z_ɪ_d_ə_n_t_ _w_ʊ_d_ _ɹ_ˈ_a_ɪ_d_ _ɹ_ˈ_ɛ_ɡ_j_ʊ_l_ɚ_l_i_ _ɪ_n_ _ɐ_ _v_ˈ_i_ə_k_ə_l_ _w_ɪ_ð_ _ɐ_ _f_ˈ_ɪ_k_s_t_ _t_ˈ_ɑ_ː_p_,_ _ˈ_i_ː_v_ə_n_ _ð_ˌ_o_ʊ_ _t_ɹ_æ_n_s_p_ˈ_æ_ɹ_ə_n_t_._
*****************************************************
RTF:		0.015886
RTF Waveform:	0.042055


✓ 语音合成完成；文件位于： /content/synth_output


In [24]:
# 8-B｜统计推理速度（从原长 Cell 拆出，便于学习）
# Model RTF：仅 Matcha 生成 Mel 的时间 / 语音时长
# Waveform RTF：Matcha + HiFi-GAN 总时间 / 语音时长
print(f"Number of ODE steps: {n_timesteps}")
print(f"Mean RTF:\t\t\t\t{np.mean(rtfs):.6f} ± {np.std(rtfs):.6f}")
print(f"Mean RTF Waveform (incl. vocoder):\t{np.mean(rtfs_w):.6f} ± {np.std(rtfs_w):.6f}")
print("输出文件：", sorted(p.name for p in Path(OUTPUT_FOLDER).iterdir()))

Number of ODE steps: 10
Mean RTF:				0.015886 ± 0.000000
Mean RTF Waveform (incl. vocoder):	0.042055 ± 0.000000
输出文件： ['0.npy', '0.wav']


### 运行结果与报错定位（v3）

- **`numpy.ufunc ... __module__`**：先使用「运行时 → 断开连接并删除运行时」，重新连接，按 0-A～0-G 运行；安装后 0-F **会自动重启一次**。这是之前版本最关键的修改。
- **`ModuleNotFoundError: matcha`**：确认 0-E 中安装命令已经成功执行，不可跳过失败的安装 Cell。
- **`UsageError: unrecognized arguments`**：Notebook 已移除 `%matplotlib inline` 的行内中文注释。
- **`AssertionError: NumPy ...`**：0-G 发现版本不一致，不要继续执行 1-B；删除运行时重新开始。
- **CUDA 不可用**：在「运行时 → 更改运行时类型」选择 T4 GPU，然后重新运行。
- **第一次下载模型**：2-B 会从官方 checkpoint 发布页下载，下载耗时因网络而异。
- **`weights_only=False`**：只加载固定官方发布来源的可信 checkpoint；不要这样加载来源不明的文件。

**测试范围**：已进行 Notebook 结构验证、全部代码 Cell 语法编译、安装/重启分支的模拟检查。无法在本地运行远端 Colab 的 apt/pip、GPU 和在线 checkpoint，因此完整实测仍需在 Colab 执行 7-B 冒烟测试。

In [29]:

# ============================================================
# 9 | Matcha-TTS 超参数对比实验
# 目标：研究 ODE Steps、Speaking Rate、Temperature 的影响
# ============================================================

import time
from pathlib import Path

# ① 实验设置：所有实验使用同一句英语
test_text = "Hello, this is a test of Matcha text to speech."
experiment_dir = Path(OUTPUT_FOLDER) / "experiments"
experiment_dir.mkdir(parents=True, exist_ok=True)

# ② 参数设置：每次只修改一个超参数
experiments = [
    # Baseline：原始默认参数
    {"name": "baseline", "steps": 10, "scale": 1.0, "temp": 0.667},

    # Experiment A：比较 ODE Solver 步数对速度和音质的影响
    {"name": "A_steps_5",  "steps": 2,  "scale": 1.0, "temp": 0.667},
    {"name": "A_steps_20", "steps": 50, "scale": 1.0, "temp": 0.667},

    # Experiment B：比较 Speaking Rate 对语音时长的影响
    {"name": "B_scale_0.8", "steps": 10, "scale": 0.5, "temp": 0.667},
    {"name": "B_scale_1.2", "steps": 10, "scale": 1.5, "temp": 0.667},

    # Experiment C：比较 Temperature 对生成随机性的影响
    {"name": "C_temp_0.3", "steps": 10, "scale": 1.0, "temp": 0.2},
    {"name": "C_temp_1.0", "steps": 10, "scale": 1.0, "temp": 2.0},
]

# ③ 保存原来的全局参数，实验结束后恢复
original_params = (n_timesteps, length_scale, temperature)
results = []

try:
    for exp in experiments:

        # 更新当前实验参数（synthesise() 使用这三个全局变量）
        n_timesteps = exp["steps"]
        length_scale = exp["scale"]
        temperature = exp["temp"]

        # 固定随机种子，尽量减少随机性对参数比较的干扰
        torch.manual_seed(42)

        # GPU 同步：避免异步执行造成计时不准确
        if device.type == "cuda":
            torch.cuda.synchronize()

        # ④ Matcha-TTS：Text → Mel-spectrogram
        start = time.perf_counter()
        output = synthesise(test_text)

        if device.type == "cuda":
            torch.cuda.synchronize()

        mel_time = time.perf_counter() - start

        # ⑤ HiFi-GAN：Mel-spectrogram → Waveform
        output["waveform"] = to_waveform(output["mel"], vocoder)

        if device.type == "cuda":
            torch.cuda.synchronize()

        total_time = time.perf_counter() - start

        # ⑥ 计算 Duration 和 RTF
        duration = output["waveform"].numel() / 22050
        mel_rtf = mel_time / duration
        total_rtf = total_time / duration

        # ⑦ 保存 WAV + Mel 特征
        save_to_folder(exp["name"], output, experiment_dir)

        # ⑧ 记录实验结果
        results.append({
            "Experiment": exp["name"],
            "ODE Steps": n_timesteps,
            "Length Scale": length_scale,
            "Temperature": temperature,
            "Duration (s)": round(duration, 3),
            "Mel RTF": round(mel_rtf, 4),
            "Total RTF": round(total_rtf, 4),
        })

        # ⑨ 直接播放当前实验的合成语音
        print(f"\n{'=' * 55}")
        print(f"Experiment: {exp['name']}")
        print(f"ODE Steps: {n_timesteps}")
        print(f"Length Scale: {length_scale}")
        print(f"Temperature: {temperature}")
        print(f"Duration: {duration:.3f} s")
        print(f"Mel RTF: {mel_rtf:.4f}")
        print(f"Total RTF: {total_rtf:.4f}")

        ipd.display(
            ipd.Audio(
                output["waveform"].numpy(),
                rate=22050
            )
        )

finally:
    # ⑩ 恢复原始参数，不影响后续原 Notebook 的推理
    n_timesteps, length_scale, temperature = original_params


# ============================================================
# 10 | 实验结果汇总
# ============================================================

import pandas as pd

df = pd.DataFrame(results)

print("\n========== Matcha-TTS 实验结果 ==========")
display(df)

# 保存实验指标为 CSV，方便后续论文分析
csv_path = experiment_dir / "experiment_results.csv"
df.to_csv(csv_path, index=False)

print("\n所有实验完成！")
print("WAV / Mel / CSV 保存目录：", experiment_dir.resolve())



Experiment: baseline
ODE Steps: 10
Length Scale: 1.0
Temperature: 0.667
Duration: 3.901 s
Mel RTF: 0.0302
Total RTF: 0.0729



Experiment: A_steps_5
ODE Steps: 2
Length Scale: 1.0
Temperature: 0.667
Duration: 3.901 s
Mel RTF: 0.0108
Total RTF: 0.0475



Experiment: A_steps_20
ODE Steps: 50
Length Scale: 1.0
Temperature: 0.667
Duration: 3.901 s
Mel RTF: 0.2883
Total RTF: 0.3245



Experiment: B_scale_0.8
ODE Steps: 10
Length Scale: 0.5
Temperature: 0.667
Duration: 1.950 s
Mel RTF: 0.0770
Total RTF: 0.1157



Experiment: B_scale_1.2
ODE Steps: 10
Length Scale: 1.5
Temperature: 0.667
Duration: 5.851 s
Mel RTF: 0.0252
Total RTF: 0.0635



Experiment: C_temp_0.3
ODE Steps: 10
Length Scale: 1.0
Temperature: 0.2
Duration: 3.901 s
Mel RTF: 0.0327
Total RTF: 0.0687



Experiment: C_temp_1.0
ODE Steps: 10
Length Scale: 1.0
Temperature: 2.0
Duration: 3.901 s
Mel RTF: 0.0341
Total RTF: 0.0703



========== Matcha-TTS 实验结果 ==========


,Experiment,ODE Steps,Length Scale,Temperature,Duration (s),Mel RTF,Total RTF
0,baseline,10,1.0,0.667,3.901,0.0302,0.0729
1,A_steps_5,2,1.0,0.667,3.901,0.0108,0.0475
2,A_steps_20,50,1.0,0.667,3.901,0.2883,0.3245
3,B_scale_0.8,10,0.5,0.667,1.950,0.0770,0.1157
4,B_scale_1.2,10,1.5,0.667,5.851,0.0252,0.0635
5,C_temp_0.3,10,1.0,0.200,3.901,0.0327,0.0687
6,C_temp_1.0,10,1.0,2.000,3.901,0.0341,0.0703



所有实验完成！
WAV / Mel / CSV 保存目录： /content/synth_output/experiments


In [30]:

# ============================================================
# ② Matcha-TTS 超参数对比实验
# 每次只修改一个超参数，其余保持 Baseline 不变
# ============================================================

experiments = [

    # --------------------------------------------------------
    # Baseline：默认参数（对照组）
    # ODE Steps = 10：默认积分精度
    # Length Scale = 1.0：正常语速
    # Temperature = 0.667：默认初始噪声强度
    # --------------------------------------------------------
    {"name": "baseline", "steps": 10, "scale": 1.0, "temp": 0.667},

    # --------------------------------------------------------
    # Experiment A：ODE Solver Steps（积分步数）
    # 研究 Flow Matching 的积分精度、推理速度和音质
    #
    # Steps ↓ 10 → 2：
    #   ODE 更新次数减少，推理更快
    #   但积分误差增大，可能出现发音不清或失真
    #
    # Steps ↑ 10 → 50：
    #   ODE 更新次数增加，积分结果通常更精细
    #   但推理更慢，音质不一定继续提升
    # --------------------------------------------------------
    {"name": "A_steps_2",  "steps": 2,  "scale": 1.0, "temp": 0.667},
    {"name": "A_steps_50", "steps": 50, "scale": 1.0, "temp": 0.667},

    # --------------------------------------------------------
    # Experiment B：Length Scale（语音时长缩放）
    # 研究 Duration Predictor 对 Speaking Rate 的影响
    #
    # Scale ↓ 1.0 → 0.5：
    #   音素持续时间缩短，语速加快
    #   预计音频长度约减半，可能出现发音过快
    #
    # Scale ↑ 1.0 → 1.5：
    #   音素持续时间延长，语速减慢
    #   预计音频长度约增加50%，可能出现拖音
    # --------------------------------------------------------
    {"name": "B_scale_0.5", "steps": 10, "scale": 0.5, "temp": 0.667},
    {"name": "B_scale_1.5", "steps": 10, "scale": 1.5, "temp": 0.667},

    # --------------------------------------------------------
    # Experiment C：Temperature（初始噪声强度）
    # 研究初始 Noise 对最终 Mel-spectrogram 的影响
    #
    # Temp ↓ 0.667 → 0.2：
    #   初始随机噪声幅度降低
    #   可能使结果更稳定，但不保证音质更高
    #
    # Temp ↑ 0.667 → 2.0：
    #   初始随机噪声幅度明显增大
    #   生成结果可能变化更明显，但也可能出现失真
    # --------------------------------------------------------
    {"name": "C_temp_0.2", "steps": 10, "scale": 1.0, "temp": 0.2},
    {"name": "C_temp_2.0", "steps": 10, "scale": 1.0, "temp": 2.0},
]
